In [1]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import warnings
warnings.filterwarnings("ignore")
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import pickle
import wandb
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

MODEL_NAME = 'VGG-16'   # <-- change per notebook
GPU_ID = 1               # <-- change to 1 for a second notebook running in parallel

os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU_ID)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device} (physical GPU {GPU_ID})")

with open('../data/data_config.pkl', 'rb') as f:
    config = pickle.load(f)

train_df      = config['train_df']
val_df        = config['val_df']
test_df       = config['test_df']
CLASSES       = config['classes']
class_weights = config['class_weights']
IMAGE_SIZE    = config['image_size']
BATCH_SIZE    = config['batch_size']

SEG_DIR = '../data/segmented_images'
assert os.path.isdir(SEG_DIR)
n_seg = len(os.listdir(SEG_DIR))
print(f"Segmented images found: {n_seg:,}")

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}  Classes: {len(CLASSES)}")
print(f"CLASSES = {CLASSES}")

Device : cuda (physical GPU 1)
Segmented images found: 112,120
Train: 80,726  Val: 8,970  Test: 22,424  Classes: 15
CLASSES = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia', 'No Finding']


In [2]:
wandb.init(
    project='xray-classification',
    name=f'03-classification-{MODEL_NAME}',
    config={
        'model': MODEL_NAME,
        'image_size': IMAGE_SIZE,
        'batch_size': BATCH_SIZE,
        'seg_source': SEG_DIR,
        'num_classes': len(CLASSES)
    }
)
print(f"✅ Wandb run started: 03-classification-{MODEL_NAME}")

wandb: Currently logged in as: chandinigunna06 (chandinigunna06-iiest-shibpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Wandb run started: 03-classification-VGG-16


In [3]:
FILENAME_COL = 'Image Index'
LABEL_COL    = 'Finding Labels'

def build_multihot(df, classes):
    label_lists = df[LABEL_COL].str.split('|')
    multihot = np.zeros((len(df), len(classes)), dtype=np.float32)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    for row_idx, labels in enumerate(label_lists):
        for lbl in labels:
            lbl = lbl.strip()
            if lbl in class_to_idx:
                multihot[row_idx, class_to_idx[lbl]] = 1.0
    return multihot

class NIHClassificationDataset(Dataset):
    def __init__(self, df, image_dir, classes, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.classes = classes
        self.transform = transform
        self.labels = build_multihot(self.df, classes)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fname = self.df.iloc[idx][FILENAME_COL]
        img = Image.open(os.path.join(self.image_dir, fname)).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx])

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = NIHClassificationDataset(train_df, SEG_DIR, CLASSES, train_transform)
val_dataset   = NIHClassificationDataset(val_df,   SEG_DIR, CLASSES, val_transform)
test_dataset  = NIHClassificationDataset(test_df,  SEG_DIR, CLASSES, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=8, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")
print(f"Label matrix shape: {train_dataset.labels.shape}")
print("Positive counts per class (train):")
for c, cnt in zip(CLASSES, train_dataset.labels.sum(axis=0)):
    print(f"  {c:20s}: {int(cnt):,}")

Train batches: 2523 | Val: 281 | Test: 701
Label matrix shape: (80726, 15)
Positive counts per class (train):
  Atelectasis         : 8,388
  Cardiomegaly        : 2,012
  Effusion            : 9,673
  Infiltration        : 14,388
  Mass                : 4,168
  Nodule              : 4,551
  Pneumonia           : 960
  Pneumothorax        : 3,828
  Consolidation       : 3,395
  Edema               : 1,673
  Emphysema           : 1,817
  Fibrosis            : 1,243
  Pleural_Thickening  : 2,455
  Hernia              : 160
  No Finding          : 43,329


In [4]:
from torch.utils.data import WeightedRandomSampler

# Vectorized: weight vector aligned to CLASSES order
weight_vec = np.array([class_weights[c] for c in CLASSES])   # shape (15,)

# Per-sample weight = max weight among the classes present (labels is 0/1 multi-hot)
sample_weights = (train_dataset.labels * weight_vec).max(axis=1)

print(f"Sample weight range: {sample_weights.min():.3f} to {sample_weights.max():.3f}")
print(f"Mean sample weight : {sample_weights.mean():.3f}")

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Rebuild train_loader with sampler instead of shuffle
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,          # <-- replaces shuffle=True
    num_workers=8,
    pin_memory=True
)

print(f"✅ Weighted sampler active — train_loader now oversamples rare classes")
print(f"   (val_loader / test_loader stay as plain sequential loaders — no sampler on eval)")

Sample weight range: 0.157 to 42.517
Mean sample weight : 1.014
✅ Weighted sampler active — train_loader now oversamples rare classes
   (val_loader / test_loader stay as plain sequential loaders — no sampler on eval)


In [5]:
import os
os.environ['HF_TOKEN'] = 'REDACTED_HF_TOKEN' 

In [6]:
import timm
import torch.nn as nn

model = timm.create_model(
    'vgg16',
    pretrained=True,
    num_classes=len(CLASSES)
)

model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"✅ VGG16 loaded → {n_params:,} parameters")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

criterion = nn.BCEWithLogitsLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)


✅ VGG16 loaded → 134,321,999 parameters


In [37]:
import torch
import gc

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

# Check available memory
for i in range(torch.cuda.device_count()):
    total = torch.cuda.get_device_properties(i).total_memory / 1024**3
    used  = torch.cuda.memory_allocated(i) / 1024**3
    free  = total - used
    print(f"GPU {i}: Total={total:.1f}GB | Used={used:.1f}GB | Free={free:.1f}GB")

GPU 0: Total=31.7GB | Used=3.2GB | Free=28.6GB


In [21]:
import torch
import gc

# Force GPU 1 only
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

# Clear everything
torch.cuda.empty_cache()
gc.collect()

# Reinitialize CUDA
torch.cuda.init()

# Check
print(f"CUDA initialized: {torch.cuda.is_initialized()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Free memory: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1024**3:.1f}GB")

CUDA initialized: True
GPU: Tesla V100-PCIE-32GB
Free memory: 28.6GB


In [30]:
# THIS MUST BE THE VERY FIRST CELL TO RUN!
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'  # Use GPU 1 only
print("GPU set to physical GPU 1")

GPU set to physical GPU 1


In [7]:
import time

CHECKPOINT_DIR = f'../checkpoints/{MODEL_NAME}'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'latest.pt')
BEST_PATH = os.path.join(CHECKPOINT_DIR, 'best.pt')

NUM_EPOCHS = 50
EARLY_STOP_PATIENCE = 7
start_epoch = 0
best_val_auc = 0.0
epochs_since_improvement = 0

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch = ckpt['epoch'] + 1
    best_val_auc = ckpt['best_val_auc']
    epochs_since_improvement = ckpt.get('epochs_since_improvement', 0)
    print(f"🔄 Resumed from epoch {start_epoch} (best val AUC so far: {best_val_auc:.4f})")
else:
    print("🆕 Starting fresh training run")


def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, desc="Train" if training else "Val"):
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            all_preds.append(torch.sigmoid(outputs).detach().cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    aucs = [roc_auc_score(all_labels[:, i], all_preds[:, i])
            for i in range(len(CLASSES)) if all_labels[:, i].sum() > 0]
    return total_loss / len(loader), np.mean(aucs)


for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()
    train_loss, train_auc = run_epoch(train_loader, training=True)
    val_loss, val_auc = run_epoch(val_loader, training=False)
    scheduler.step(val_auc/home/lhotse1/student/btech2023/achanta/miniconda3/envs/cv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

GPU    : Tesla V100-PCIE-32GB
Free   : 31.7GB
Train  : 80,726
Classes: 15
✅ Ready!
)
    elapsed = time.time() - t0

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} AUC: {train_auc:.4f} | "
          f"Val Loss: {val_loss:.4f} AUC: {val_auc:.4f} | "
          f"{elapsed:.0f}s")

    wandb.log({
        'epoch': epoch + 1, 'train_loss': train_loss, 'train_auc': train_auc,
        'val_loss': val_loss, 'val_auc': val_auc, 'lr': optimizer.param_groups[0]['lr']
    })

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        epochs_since_improvement = 0
        torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'val_auc': val_auc}, BEST_PATH)
        print(f"  💾 New best model saved (val AUC: {val_auc:.4f})")
    else:
        epochs_since_improvement += 1

    torch.save({
        'epoch': epoch, 'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(), 'scheduler_state': scheduler.state_dict(),
        'best_val_auc': best_val_auc, 'epochs_since_improvement': epochs_since_improvement
    }, CHECKPOINT_PATH)

    if epochs_since_improvement >= EARLY_STOP_PATIENCE:
        print(f"\n⏹️ Early stopping — no improvement for {EARLY_STOP_PATIENCE} epochs")
        break
  
wandb.log({'final_best_val_auc': best_val_auc})
print(f"\n✅ Training complete! Best val AUC: {best_val_auc:.4f}")

🆕 Starting fresh training run


Val: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 281/281 [00:15<00:00, 17.59it/s]


Epoch 1/50 | Train Loss: 0.3018 AUC: 0.7474 | Val Loss: 0.2475 AUC: 0.7587 | 475s
  💾 New best model saved (val AUC: 0.7587)


Val: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 281/281 [00:14<00:00, 18.87it/s]


Epoch 2/50 | Train Loss: 0.2541 AUC: 0.8378 | Val Loss: 0.2513 AUC: 0.7574 | 469s


...

Epoch 6/50 | Train Loss: 0.0788 AUC: 0.9819 | Val Loss: 0.3073 AUC: 0.7143 | 472s


Val: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 281/281 [00:14<00:00, 18.76it/s]


Epoch 7/50 | Train Loss: 0.0643 AUC: 0.9867 | Val Loss: 0.3213 AUC: 0.7024 | 472s


Val: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 281/281 [00:14<00:00, 18.83it/s]


Epoch 8/50 | Train Loss: 0.0464 AUC: 0.9918 | Val Loss: 0.3423 AUC: 0.7142 | 470s

⏹️ Early stopping — no improvement for 7 epochs

✅ Training complete! Best val AUC: 0.7587


In [8]:
# Load the best checkpoint (epoch 1's weights, which had the best val AUC)
best_ckpt = torch.load(BEST_PATH, map_location=device, weights_only=False)
model.load_state_dict(best_ckpt['model_state'])
model.eval()

print(f"Loaded best checkpoint from epoch {best_ckpt['epoch']+1} (val AUC: {best_ckpt['val_auc']:.4f})")

test_loss, test_auc = run_epoch(test_loader, training=False)

# Per-class test AUC breakdown
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Final test eval"):
        images = images.to(device)
        outputs = model(images)
        all_preds.append(torch.sigmoid(outputs).cpu().numpy())
        all_labels.append(labels.numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

per_class_auc = {}
for i, c in enumerate(CLASSES):
    if all_labels[:, i].sum() > 0:
        per_class_auc[c] = roc_auc_score(all_labels[:, i], all_preds[:, i])
    else:
        per_class_auc[c] = None

print(f"\n{'='*50}")
print(f"TEST RESULTS — {MODEL_NAME}")
print(f"{'='*50}")
print(f"Overall Test Loss: {test_loss:.4f}")
print(f"Overall Test AUC : {test_auc:.4f}")
print(f"\nPer-class Test AUC:")
for c, auc in sorted(per_class_auc.items(), key=lambda x: -(x[1] or 0)):
    print(f"  {c:20s}: {auc:.4f}" if auc is not None else f"  {c:20s}: N/A")

wandb.log({
    'test_loss': test_loss,
    'test_auc': test_auc,
    **{f'test_auc_{c}': v for c, v in per_class_auc.items() if v is not None}
})

# Save results into a shared pickle for the comparison notebook
import pickle
RESULTS_PATH = '../data/class_results.pkl'

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, 'rb') as f:
        all_results = pickle.load(f)
else:
    all_results = {}

all_results[MODEL_NAME] = {
    'val_auc': best_ckpt['val_auc'],
    'test_auc': test_auc,
    'test_loss': test_loss,
    'per_class_auc': per_class_auc,
    'best_epoch': best_ckpt['epoch'] + 1
}

with open(RESULTS_PATH, 'wb') as f:
    pickle.dump(all_results, f)

print(f"\n✅ Results saved to {RESULTS_PATH}")
print(f"Models recorded so far: {list(all_results.keys())}")

wandb.finish()

Loaded best checkpoint from epoch 1 (val AUC: 0.7587)


Final test eval: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 701/701 [00:36<00:00, 19.17it/s]


TEST RESULTS — VGG-16
Overall Test Loss: 0.2471
Overall Test AUC : 0.7494

Per-class Test AUC:
  Cardiomegaly        : 0.8838
  Edema               : 0.8577
  Effusion            : 0.8313
  Pneumothorax        : 0.8240
  Emphysema           : 0.7948
  Mass                : 0.7519
  Atelectasis         : 0.7501
  Consolidation       : 0.7457
  No Finding          : 0.7379
  Pleural_Thickening  : 0.7183
  Fibrosis            : 0.7054
  Hernia              : 0.7014
  Pneumonia           : 0.6509
  Infiltration        : 0.6456
  Nodule              : 0.6419

✅ Results saved to ../data/class_results.pkl
Models recorded so far: ['VGG-16']


epoch,▁▂▃▄▅▆▇█
final_best_val_auc,▁
lr,███▃▃▃▁▁
test_auc,▁
test_auc_Atelectasis,▁
test_auc_Cardiomegaly,▁
test_auc_Consolidation,▁
test_auc_Edema,▁
test_auc_Effusion,▁
test_auc_Emphysema,▁
+14,...
